# Cartographic furniture & styling (3-D)

The 3-D (PyVista) tier draws the same cartographic furniture a 2-D map does, plus the lighting and export a
true-3-D scene needs. This notebook walks through the verbs that frame, annotate, light and export a scene —
`domain`, `graticule`, `scale_bar`, `north_arrow`, `attribution`, `labels`, `sun`/`shadows`/`ambient_occlusion`,
`bivariate`, `basemap`, `export_vector` and scene bundles — each explained before it is shown.

Every scene here is built **off-screen** (headless), with synthetic data, so the notebook runs anywhere with no
files and, where a step needs the network, guards it.

In [1]:
import numpy as np

from digitalearth.base.sources import get_source
from digitalearth.three_d import Scene3D

# A synthetic geographic relief over Europe, so the framing and the graticule have lon/lat to work with.
lon = np.linspace(-10.0, 30.0, 40)
lat = np.linspace(35.0, 60.0, 30)
relief = np.add.outer(np.cos(np.radians(lat)), np.sin(np.radians(lon)))
source = get_source(relief, x=lon, y=lat)

scene = Scene3D(off_screen=True, crs=4326)
scene.terrain(source, z_exaggeration=8.0)
scene.screenshot().shape

(768, 1024, 3)

## Framing — `domain`

`domain` points the camera at a named region or an explicit `(west, south, east, north)` box, reprojected into
the scene's display CRS through pyramids. It frames the view without flattening the relief.

In [2]:
scene.domain("europe")
scene.screenshot().shape

(768, 1024, 3)

## Graticule — `graticule`

A latitude/longitude grid, labelled along its edges. `lon_step`/`lat_step` (or `spacing` for a square grid) set
the interval; the lines are reprojected into the display CRS so the grid lies on the data.

In [3]:
scene.graticule(spacing=10.0)
scene.layer_ids

['graticule-1', 'terrain-1']

## Scale bar, north arrow, attribution

A ground-distance scale bar (which switches the view to orthographic — the only projection it is honest in), a
camera-aware north arrow, and a corner credit line, so a rendered scene carries its own context.

In [4]:
scene.scale_bar()
scene.north_arrow()
scene.attribution("© Natural Earth")
scene.screenshot().shape

(768, 1024, 3)

## Labels — `labels`

Label a feature collection from one of its columns, each label on a filled halo so it reads against the scene.

In [5]:
import geopandas as gpd
from shapely.geometry import Point

cities = gpd.GeoDataFrame(
    {"name": ["Paris", "Berlin", "Madrid"]},
    geometry=[Point(2.35, 48.85), Point(13.4, 52.52), Point(-3.7, 40.42)],
    crs=4326,
)
scene.labels(cities, "name")
scene.layer_ids

['graticule-1', 'terrain-1', 'labels-1']

## Lighting — `sun`, `shadows`, `ambient_occlusion`

Rasterization-grade relief shading: a directional sun placed by azimuth and elevation, plus cast shadows and
ambient occlusion. (This is not path tracing — VTK is a rasterizer — but it gives relief real depth.)

In [6]:
scene.sun(azimuth=135.0, elevation=25.0)
scene.shadows()
scene.ambient_occlusion()
scene.screenshot().shape

2026-10-30 00:33:11.279 (   1.810s) [    7F6CFB872740]   vtkShaderProgram.cxx:1282   ERR| vtkShaderProgram (0x55f088e70050): 1: #version 150
2: #ifdef GL_ES
3: #ifdef GL_FRAGMENT_PRECISION_HIGH
4: precision highp float;
5: precision highp sampler2D;
6: precision highp sampler3D;
7: #else
8: precision mediump float;
9: precision mediump sampler2D;
10: precision mediump sampler3D;
11: #endif
12: #define texelFetchBuffer texelFetch
13: #define texture1D texture
14: #define texture2D texture
15: #define texture3D texture
16: #else // GL_ES
17: #define highp
18: #define mediump
19: #define lowp
20: #if __VERSION__ >= 150
21: #define texelFetchBuffer texelFetch
22: #define texture1D texture
23: #define texture2D texture
24: #define texture3D texture
25: #endif
26: #endif // GL_ES
27: #define varying in
28: 
29: 
30: // SPDX-FileCopyrightText: Copyright (c) Ken Martin, Will Schroeder, Bill Lorensen
31: // SPDX-License-Identifier: BSD-3-Clause
32: // Template for the polydata mappers fragment 

:Impl
135: 
136:   // Set gl_FragDepth here (gl_FragCoord.z by default)
137:   //VTK::Depth::Impl
138: 
139:   // Early depth peeling abort:
140:   //VTK::DepthPeeling::PreColor
141: 
142:   // Apple Bug
143:   //VTK::PrimID::Impl
144: 
145:   //VTK::Clip::Impl
146: 
147:   //VTK::ValuePass::Impl
148: 
149:     vec3 ambientColor = ambientIntensity * ambientColorUniform;
150:   vec3 diffuseColor = diffuseIntensity * diffuseColorUniform;
151:   float opacity = opacityUniform;
152: 
153: 
154:   //VTK::Edges::Impl
155: 
156:   // Generate the normal if we are not passed in one
157:   //VTK::Normal::Impl
158: 
159:   //VTK::PBR::Impl
160: 
161:   vec2 factor0 = calcShadow(vertexVC, shadowMap0, shadowTransform0, shadowAttenuation0, shadowParallel0, shadowNearZ0, shadowFarZ0);
162:   fragOutput0 = vec4(ambientColor + diffuseColor, opacity);
163:   //VTK::Light::Impl
164: 
165: 
166: 
167:   //VTK::TCoord::Impl
168: 
169:   if (fragOutput0.a <= 0.0)
170:     {
171:     discard;
172:     }
173

: 
174:   //VTK::DepthPeeling::Impl
175: 
176:   //VTK::Picking::Impl
177: 
178:   // handle coincident offsets
179:   //VTK::Coincident::Impl
180: }

2026-10-30 00:33:11.281 (   1.812s) [    7F6CFB872740]   vtkShaderProgram.cxx:1283   ERR| vtkShaderProgram (0x55f088e70050): 0:161(28): error: `vertexVC' undeclared
0:161(17): error: no matching function for call to `calcShadow(error, sampler2D, mat4, float, int, float, float)'; candidates are:
0:161(17): error:    vec2 calcShadow(vec4, sampler2D, mat4, float, int, float, float)

2026-10-30 00:33:11.283 (   1.814s) [    7F6CFB872740]vtkOpenGLPolyDataMapper:2771   ERR| vtkOpenGLPolyDataMapper (0x55f087e5f240): Could not set shader program
ERROR:root:Could not set shader program


2026-10-30 00:33:11.283 (   1.815s) [    7F6CFB872740]vtkOpenGLVertexArrayObj:262    ERR| vtkOpenGLVertexArrayObject (0ERROR:root:attempt to add attribute without a bound program for attribute vertexMC
x55f087fe2fd0): attempt to add attribute without a bound program for attribute vertexMC


ERROR:root:Error setting 'vertexMC' in shader VAO.
2026-10-30 00:33:11.284 (   1.816s) [    7F6CFB872740]vtkOpenGLVertexBufferOb:267    ERR| vtkOpenGLVertexBufferObjectGroup (0x55f087c41760): Error setting 'vertexMC' in shader VAO.


(768, 1024, 3)

## Bivariate styling — `bivariate`

Two variables encoded as one colour from an n×n matrix: each variable is cut into classes by the shared
classifier, and the class pair picks a colour. The classification is engine-neutral; the tier drapes the
resulting image on a ground plane.

In [7]:
a = np.add.outer(np.linspace(0.0, 1.0, 30), np.linspace(0.0, 1.0, 40))
b = np.add.outer(np.linspace(1.0, 0.0, 30), np.linspace(0.0, 1.0, 40))
bivariate_scene = Scene3D(off_screen=True)
bivariate_scene.bivariate(a, b)
shape = bivariate_scene.screenshot().shape
bivariate_scene.close()
shape

(768, 1024, 3)

## Basemap — `basemap`

Web-map tiles draped under the scene as ground cover. Tiles are Web Mercator, so draw the scene in EPSG:3857
for them to line up. This cell reaches the network, so it is guarded — it renders when a tile server is
reachable and reports a skip otherwise.

In [8]:
mercator = Scene3D(off_screen=True, crs=3857)
try:
    mercator.basemap("CartoDB.Positron", bbox=(-10.0, 35.0, 30.0, 60.0), zoom=3)
    print("basemap drawn:", mercator.layer_ids)
except Exception as error:  # offline, or the tile server is unreachable
    print("basemap skipped (no network):", type(error).__name__)
finally:
    mercator.close()

basemap drawn: ['basemap']


## Export — vector graphics and scene bundles

`export_vector` writes the view as SVG/PDF/EPS through VTK's GL2PS exporter (a high-resolution raster master is
`screenshot(scale=N)`). `save_bundle` writes the whole scene to a versioned zip — but a bundle stores a
*description*, so it needs path- or URL-backed data; a scene built from in-memory arrays is refused, which this
cell shows.

In [9]:
import os
import tempfile

folder = tempfile.mkdtemp()
scene.export_vector(os.path.join(folder, "scene.svg"))
print("vector export written:", os.path.exists(os.path.join(folder, "scene.svg")))

try:
    scene.save_bundle(os.path.join(folder, "scene.dez"))
except ValueError as error:
    print("in-memory scene is not bundleable:", str(error).split(";")[0][:72], "...")

2026-10-30 00:33:12.017 (   2.549s) [    7F6CFB872740]     vtkOpenGLState.cxx:627   WARN| Attempt to set draw buffers from a Framebuffer Object that is not bound.
2026-10-30 00:33:12.037 (   2.569s) [    7F6CFB872740]     vtkOpenGLState.cxx:627   WARN| Attempt to set draw buffers from a Framebuffer Object that is not bound.
2026-10-30 00:33:12.118 (   2.649s) [    7F6CFB872740]   vtkShaderProgram.cxx:1282   ERR| vtkShaderProgram (0x55f08ce89010): 1: #version 150
2: #ifdef GL_ES
3: #ifdef GL_FRAGMENT_PRECISION_HIGH
4: precision highp float;
5: precision highp sampler2D;
6: precision highp sampler3D;
7: #else
8: precision mediump float;
9: precision mediump sampler2D;
10: precision mediump sampler3D;
11: #endif
12: #define texelFetchBuffer texelFetch
13: #define texture1D texture
14: #define texture2D texture
15: #define texture3D texture
16: #else // GL_ES
17: #define highp
18: #define mediump
19: #define lowp
20: #if __VERSION__ >= 150
21: #define texelFetchBuffer texelFetch
22: #defin

ERROR:root:0:163(28): error: `vertexVC' undeclared
0:163(17): error: no matching function for call to `calcShadow(error, sampler2D, mat4, float, int, float, float)'; candidates are:
0:163(17): error:    vec2 calcShadow(vec4, sampler2D, mat4, float, int, float, float)
2026-10-30 00:33:12.119 (   2.650s) [    7F6CFB872740]   vtkShaderProgram.cxx:1283   ERR| vtkShaderProgram (0x55f08ce89010): 0:163(28): error: `vertexVC' undeclared
0:163(17): error: no matching function for call to `calcShadow(error, sampler2D, mat4, float, int, float, float)'; candidates are:
0:163(17): error:    vec2 calcShadow(vec4, sampler2D, mat4, float, int, float, float)



ERROR:root:Could not set shader program


2026-10-30 00:33:12.120 (   2.651s) [    7F6CFB872740]vtkOpenGLPolyDataMapper:2771   ERR| vtkOpenGLPolyDataMapper (0x55f087e5f240): Could not set shader program
2026-10-30 00:33:12.120 (   2.652s) [    7F6CFB872740]vtkOpenGLVertexArrayObj:262    ERR| vtkOpenGLVertexArrayObject (0x55f087fe2fd0): attempt to add attribute without a bound program for attribute vertexMC
ERROR:root:attempt to add attribute without a bound program for attribute vertexMC


2026-10-30 00:33:12.121 (   2.652s) [    7F6CFB872740]vtkOpenGLVertexBufferOb:267    ERR| vtkOpenGLVertexBufferObjectGroup (0x55f087c41760): Error setting 'vertexMC' in shader VAO.
ERROR:root:Error setting 'vertexMC' in shader VAO.


...

VTK::ValuePass::Dec
45: 
46: // picking support
47: //VTK::Picking::Dec
48: 
49: // Surface with edges on GLES 3.0
50: //VTK::EdgesGLES30::Dec
51: 
52: // PointSize on GLES 3.0
53: //VTK::PointSizeGLES30::Dec
54: 
55: // LineWidth on GLES 3.0
56: //VTK::LineWidthGLES30::Dec
57: 
58: void main()
59: {
60:   //VTK::CustomBegin::Impl
61: 
62:   //VTK::PointSizeGLES30::Impl
63: 
64:   //VTK::Color::Impl
65: 
66:   //VTK::Normal::Impl
67: 
68:   //VTK::TCoord::Impl
69: 
70:   //VTK::Clip::Impl
71: 
72:   //VTK::PrimID::Impl
73: 
74:     gl_Position = MCDCMatrix * vertexMC;
75: 
76: 
77:   //VTK::LineWidthGLES30::Impl
78: 
79:   //VTK::ValuePass::Impl
80: 
81:   //VTK::Light::Impl
82: 
83:   //VTK::Picking::Impl
84: 
85:   //VTK::CustomEnd::Impl
86: 
87:   //VTK::EdgesGLES30::Impl
88: }

ERROR:root:
2026-10-30 00:33:12.993 (   3.524s) [    7F6CFB872740]   vtkShaderProgram.cxx:1283   ERR| vtkShaderProgram (0x55f08ce89010): 


ERROR:root:Could not set shader program
2026-10-30 00:33:12.993 (   3.525s) [    7F6CFB872740]vtkOpenGLPolyDataMapper:2771   ERR| vtkOpenGLPolyDataMapper (0x55f087e5f240): Could not set shader program


ERROR:root:attempt to add attribute without a bound program for attribute vertexMC
2026-10-30 00:33:12.994 (   3.525s) [    7F6CFB872740]vtkOpenGLVertexArrayObj:262    ERR| vtkOpenGLVertexArrayObject (0x55f087fe2fd0): attempt to add attribute without a bound program for attribute vertexMC


ERROR:root:Error setting 'vertexMC' in shader VAO.
2026-10-30 00:33:12.995 (   3.526s) [    7F6CFB872740]vtkOpenGLVertexBufferOb:267    ERR| vtkOpenGLVertexBufferObjectGroup (0x55f087c41760): Error setting 'vertexMC' in shader VAO.


vector export written: True
in-memory scene is not bundleable: save_bundle(): source 'terrain-1' is 'object:2a85425bfe17:terrain-1', wh ...


## One-call framing — `quickmap(backend="3d")`

`quickmap` wires the framing (and, for an EPSG:3857 scene, the basemap) into one call. Guarded, since the
input shape and any basemap fetch depend on the environment.

In [10]:
from digitalearth import quickmap

try:
    quick = quickmap(relief, backend="3d", domain="europe")
    print("quickmap 3-D layers:", quick.layer_ids)
    quick.close()
except Exception as error:
    print("quickmap demo skipped:", type(error).__name__)

scene.close()

quickmap demo skipped: TypeError
